# Mount Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install a specific version with CUDA 11.8 support
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118


Looking in indexes: https://download.pytorch.org/whl/cu118


In [ ]:
import torch
print(torch.__version__)
print("CUDA Available:", torch.cuda.is_available())


2.5.0+cpu
CUDA Available: False


#Import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

#subnetwork for MFCC input processing

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# MFCCNet
class MFCCNet(nn.Module):
    def __init__(self, input_dim):
        super(MFCCNet, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.lstm = nn.LSTM(128, 128, batch_first=True, bidirectional=True)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(128 * 2, 128)  # Bidirectional LSTM doubles the output size
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x, _ = self.lstm(x)
        x = self.flatten(x)
        x = self.fc(x)
        x = self.dropout(x)
        return x

# CQCCNet
class CQCCNet(nn.Module):
    def __init__(self, input_dim):
        super(CQCCNet, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.lstm = nn.LSTM(128, 128, batch_first=True, bidirectional=True)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(128 * 2, 128)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x, _ = self.lstm(x)
        x = self.flatten(x)
        x = self.fc(x)
        x = self.dropout(x)
        return x

# PitchNet with Conv1D
class PitchNet(nn.Module):
    def __init__(self, input_dim):
        super(PitchNet, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=32, kernel_size=3, padding=1)
        self.gru = nn.GRU(32, 64, batch_first=True, bidirectional=True)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(64 * 2, 64)  # Bidirectional GRU doubles the output size
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x, _ = self.gru(x)
        x = self.flatten(x)
        x = self.fc(x)
        x = self.dropout(x)
        return x

# EnergyNet with Conv1D
class EnergyNet(nn.Module):
    def __init__(self, input_dim):
        super(EnergyNet, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * input_dim, 64)  # Adjust the size based on input reshaping
        self.fc2 = nn.Linear(64, 128)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        return x

# EmbeddingNet
class EmbeddingNet(nn.Module):
    def __init__(self, input_dim):
        super(EmbeddingNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 128)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        return x

# Fusion Model
class FusionModel(nn.Module):
    def __init__(self, mfcc_dim, cqcc_dim, pitch_dim, energy_dim, embedding_dim):
        super(FusionModel, self).__init__()
        self.mfcc_net = MFCCNet(mfcc_dim)
        self.cqcc_net = CQCCNet(cqcc_dim)
        self.pitch_net = PitchNet(pitch_dim)
        self.energy_net = EnergyNet(energy_dim)
        self.embedding_net = EmbeddingNet(embedding_dim)

        self.fc_fusion = nn.Linear(128 + 128 + 64 + 128 + 128, 256)  # Adjust dimensions as needed
        self.attention = nn.MultiheadAttention(embed_dim=256, num_heads=4)
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, 64)
        self.output = nn.Linear(64, 1)  # Binary classification

    def forward(self, mfcc, cqcc, pitch, energy, embedding):
        mfcc_out = self.mfcc_net(mfcc)
        cqcc_out = self.cqcc_net(cqcc)
        pitch_out = self.pitch_net(pitch)
        energy_out = self.energy_net(energy)
        embedding_out = self.embedding_net(embedding)

        # Concatenate all branch outputs
        fused = torch.cat((mfcc_out, cqcc_out, pitch_out, energy_out, embedding_out), dim=1)

        # Attention layer
        fused = fused.unsqueeze(1)  # Expand for attention input
        attn_out, _ = self.attention(fused, fused, fused)
        attn_out = attn_out.squeeze(1)  # Remove expanded dimension

        # Fusion dense layers
        x = F.relu(self.fc_fusion(attn_out))
        x = F.dropout(x, 0.4)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, 0.3)
        x = F.relu(self.fc2(x))
        output = torch.sigmoid(self.output(x))

        return output

# Instantiate the model with example input dimensions
model = FusionModel(mfcc_dim=13, cqcc_dim=60, pitch_dim=10, energy_dim=10, embedding_dim=128)
print(model)

# Function to count total parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Print the total number of parameters
print(f"Total number of parameters: {count_parameters(model):,}")


FusionModel(
  (mfcc_net): MFCCNet(
    (conv1): Conv1d(13, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
    (flatten): Flatten(start_dim=1, end_dim=-1)
    (fc): Linear(in_features=256, out_features=128, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (cqcc_net): CQCCNet(
    (conv1): Conv1d(60, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
    (flatten): Flatten(start_dim=1, end_dim=-1)
    (fc): Linear(in_features=256, out_features=128, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (pitch_net): PitchNet(
    (conv1): Conv1d(10, 32, kernel_size=(3,), stride=(1,), padding=(1,))
    (gru): GRU(32, 64, batch_first=True, bidirectional=True)
    (flatten): Flatten(start_dim=1, en

#Timestamps analysis for MFCC

## Train

In [ ]:
import os
import zipfile
# Specify the path to the zipped audio folder in Google Drive
zip_path = '/content/drive/MyDrive/EE8223 Group Project/ASV_MFCC_Features/Archive.zip'  # Update with your actual zip path

# Extract the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/mfcc_extracted')

In [ ]:
import numpy as np
import os
import zipfile


# Example paths (update these with your actual paths)
mfcc_path = '/content/mfcc_extracted/train'


# Function to load MFCC and CQCC data
def load_data(data_path):
    data_lengths = []
    for file_name in os.listdir(data_path):
        if file_name.endswith('.npy'):  # Assumes data is saved as NumPy arrays
            data = np.load(os.path.join(data_path, file_name))
            data_lengths.append(data.shape[1])  # Extract the number of time frames
    return data_lengths

# Load and get data lengths
mfcc_lengths = load_data(mfcc_path)

# Calculate statistics for MFCC timeframes (train)
mfcc_mean_length_train = np.mean(mfcc_lengths)
mfcc_std_dev_length_train = np.std(mfcc_lengths)
mfcc_fixed_frame_length_train = int(mfcc_mean_length_train )


# Display statistics for MFCC and CQCC data
print("MFCC Data Overview (train):")
print(f"Total samples: {len(mfcc_lengths)}")
print(f"Min length: {np.min(mfcc_lengths)}")
print(f"Max length: {np.max(mfcc_lengths)}")
print(f"Mean length: {np.mean(mfcc_lengths):.2f}")
print(f"Standard Deviation: {np.std(mfcc_lengths):.2f}\n")

# Example path to save in Google Drive
mfcc_file_path_train = '/content/drive/MyDrive/testing folder for deep learning/mfcc_fixed_frame_length_train.txt'

# Save the fixed frame length to a text file in Google Drive
with open(mfcc_file_path_train, "w") as mfcc_file:
    mfcc_file.write(str(mfcc_fixed_frame_length_train))

print("Fixed frame length for MFCC (train) saved successfully in Google Drive!")


MFCC Data Overview (train):
Total samples: 7027
Min length: 23
Max length: 400
Mean length: 108.00
Standard Deviation: 44.80



FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/testing folder for deep learning/mfcc_fixed_frame_length_train.txt'

##Validation

In [ ]:
import numpy as np
import os

# Example paths (update these with your actual paths)
mfcc_path = '/content/mfcc_extracted/val'


# Function to load MFCC and CQCC data
def load_data(data_path):
    data_lengths = []
    for file_name in os.listdir(data_path):
        if file_name.endswith('.npy'):  # Assumes data is saved as NumPy arrays
            data = np.load(os.path.join(data_path, file_name))
            data_lengths.append(data.shape[1])  # Extract the number of time frames
    return data_lengths

# Load and get data lengths
mfcc_lengths = load_data(mfcc_path)
# Calculate statistics for MFCC timeframes (validation)
mfcc_mean_length_val = np.mean(mfcc_lengths)
mfcc_std_dev_length_val = np.std(mfcc_lengths)
mfcc_fixed_frame_length_val = int(mfcc_mean_length_val )


# Display statistics for MFCC and CQCC data
print("MFCC Data Overview (val):")
print(f"Total samples: {len(mfcc_lengths)}")
print(f"Min length: {np.min(mfcc_lengths)}")
print(f"Max length: {np.max(mfcc_lengths)}")
print(f"Mean length: {np.mean(mfcc_lengths):.2f}")
print(f"Standard Deviation: {np.std(mfcc_lengths):.2f}\n")

# Example path to save in Google Drive
mfcc_file_path_val = '/content/drive/MyDrive/testing folder for deep learning/mfcc_fixed_frame_length_val.txt'

# Save the fixed frame length to a text file in Google Drive
with open(mfcc_file_path_val, "w") as mfcc_file:
    mfcc_file.write(str(mfcc_fixed_frame_length_val))

print("Fixed frame length for MFCC (val) saved successfully in Google Drive!")


MFCC Data Overview (val):
Total samples: 1505
Min length: 23
Max length: 331
Mean length: 108.65
Standard Deviation: 45.52



FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/testing folder for deep learning/mfcc_fixed_frame_length_val.txt'

#Test

In [ ]:
import numpy as np
import os

# Example paths (update these with your actual paths)
mfcc_path = '/content/mfcc_extracted/test'


# Function to load MFCC and CQCC data
def load_data(data_path):
    data_lengths = []
    for file_name in os.listdir(data_path):
        if file_name.endswith('.npy'):  # Assumes data is saved as NumPy arrays
            data = np.load(os.path.join(data_path, file_name))
            data_lengths.append(data.shape[1])  # Extract the number of time frames
    return data_lengths

# Load and get data lengths
mfcc_lengths = load_data(mfcc_path)
# Calculate statistics for MFCC timeframes (test)
mfcc_mean_length_test = np.mean(mfcc_lengths)
mfcc_std_dev_length_test = np.std(mfcc_lengths)
mfcc_fixed_frame_length_test = int(mfcc_mean_length_test )


# Display statistics for MFCC and CQCC data
print("MFCC Data Overview (test):")
print(f"Total samples: {len(mfcc_lengths)}")
print(f"Min length: {np.min(mfcc_lengths)}")
print(f"Max length: {np.max(mfcc_lengths)}")
print(f"Mean length: {np.mean(mfcc_lengths):.2f}")
print(f"Standard Deviation: {np.std(mfcc_lengths):.2f}\n")

# Example path to save in Google Drive
mfcc_file_path_test = '/content/drive/MyDrive/testing folder for deep learning/mfcc_fixed_frame_length_test.txt'

# Save the fixed frame length to a text file in Google Drive
with open(mfcc_file_path_test, "w") as mfcc_file:
    mfcc_file.write(str(mfcc_fixed_frame_length_test))

print("Fixed frame length for MFCC (test) saved successfully in Google Drive!")


MFCC Data Overview (test):
Total samples: 1507
Min length: 19
Max length: 341
Mean length: 97.98
Standard Deviation: 47.71

Fixed frame length for MFCC (test) saved successfully in Google Drive!


#Timestamp analysis for CQCC

##train

In [ ]:
import os
import zipfile
# Specify the path to the zipped audio folder in Google Drive
zip_path = '/content/drive/MyDrive/EE8223 Group Project/ASV_CQCC_Features/Archive.zip'  # Update with your actual zip path

# Extract the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/cqcc_extracted')

In [ ]:
import os
import numpy as np
from scipy.io import loadmat

# Example path (update this with your actual path)
cqcc_path = '/content/cqcc_extracted/train_wav'

# Function to load and count CQCC data from .mat files
def load_and_count_cqcc_data(data_path):
    total_files = len([f for f in os.listdir(data_path) if f.endswith('.mat')])
    processed_files = 0
    valid_files = 0
    data_lengths = []

    for file_name in os.listdir(data_path):
        if file_name.endswith('.mat'):  # Check if the file is a .mat file
            processed_files += 1
            try:
                mat_data = loadmat(os.path.join(data_path, file_name))
                if 'cqcc_features' in mat_data:  # Access 'cqcc_features' key
                    valid_files += 1
                    cqcc_features = mat_data['cqcc_features']
                    data_lengths.append(cqcc_features.shape[1])  # Extract the number of time frames
                else:
                    print(f"Warning: 'cqcc_features' key not found in {file_name}")
            except Exception as e:
                print(f"Error processing {file_name}: {e}")

    print(f"Total .mat files found: {total_files}")
    print(f"Processed {processed_files} files.")
    print(f"Valid CQCC files containing 'cqcc_features' key: {valid_files}")
    return data_lengths

# Load and get data lengths for CQCC
cqcc_lengths = load_and_count_cqcc_data(cqcc_path)



# Display statistics for CQCC data
if cqcc_lengths:
    cqcc_mean_length_train = np.mean(cqcc_lengths)
    cqcc_std_dev_length_train = np.std(cqcc_lengths)
    cqcc_fixed_frame_length_train = int(cqcc_mean_length_train + cqcc_std_dev_length_train)

# Display statistics for CQCC data
if cqcc_lengths:
    print("\nCQCC Data Overview (train):")
    print(f"Total samples: {len(cqcc_lengths)}")
    print(f"Min length: {np.min(cqcc_lengths)}")
    print(f"Max length: {np.max(cqcc_lengths)}")
    print(f"Mean length: {np.mean(cqcc_lengths):.2f}")
    print(f"Standard Deviation: {np.std(cqcc_lengths):.2f}")
    # Example path to save in Google Drive
    cqcc_file_path_train = '/content/drive/MyDrive/testing folder for deep learning/cqcc_fixed_frame_length_train.txt'

    # Save the fixed frame length to a text file in Google Drive
    with open(cqcc_file_path_train, "w") as cqcc_file:
        cqcc_file.write(str(cqcc_fixed_frame_length_train))

    print("Fixed frame length for CQCC (train) saved successfully in Google Drive!")
else:
    print("No valid CQCC data found.")


Total .mat files found: 7027
Processed 7027 files.
Valid CQCC files containing 'cqcc_features' key: 7027

CQCC Data Overview (train):
Total samples: 7027
Min length: 85
Max length: 1497
Mean length: 403.07
Standard Deviation: 167.99
Fixed frame length for CQCC (train) saved successfully in Google Drive!


##val

In [ ]:
import os
import numpy as np
from scipy.io import loadmat

# Example path (update this with your actual path)
cqcc_path = '/content/cqcc_extracted/val_wav'

# Function to load and count CQCC data from .mat files
def load_and_count_cqcc_data(data_path):
    total_files = len([f for f in os.listdir(data_path) if f.endswith('.mat')])
    processed_files = 0
    valid_files = 0
    data_lengths = []

    for file_name in os.listdir(data_path):
        if file_name.endswith('.mat'):  # Check if the file is a .mat file
            processed_files += 1
            try:
                mat_data = loadmat(os.path.join(data_path, file_name))
                if 'cqcc_features' in mat_data:  # Access 'cqcc_features' key
                    valid_files += 1
                    cqcc_features = mat_data['cqcc_features']
                    data_lengths.append(cqcc_features.shape[1])  # Extract the number of time frames
                else:
                    print(f"Warning: 'cqcc_features' key not found in {file_name}")
            except Exception as e:
                print(f"Error processing {file_name}: {e}")

    print(f"Total .mat files found: {total_files}")
    print(f"Processed {processed_files} files.")
    print(f"Valid CQCC files containing 'cqcc_features' key: {valid_files}")
    return data_lengths

# Load and get data lengths for CQCC
cqcc_lengths = load_and_count_cqcc_data(cqcc_path)

# Display statistics for CQCC data
if cqcc_lengths:
    cqcc_mean_length_val = np.mean(cqcc_lengths)
    cqcc_std_dev_length_val = np.std(cqcc_lengths)
    cqcc_fixed_frame_length_val = int(cqcc_mean_length_val + cqcc_std_dev_length_val)
    print("\nCQCC Data Overview (val):")
    print(f"Total samples: {len(cqcc_lengths)}")
    print(f"Min length: {np.min(cqcc_lengths)}")
    print(f"Max length: {np.max(cqcc_lengths)}")
    print(f"Mean length: {np.mean(cqcc_lengths):.2f}")
    print(f"Standard Deviation: {np.std(cqcc_lengths):.2f}")

    # Example path to save in Google Drive
    cqcc_file_path_val = '/content/drive/MyDrive/testing folder for deep learning/cqcc_fixed_frame_length_val.txt'

    # Save the fixed frame length to a text file in Google Drive
    with open(cqcc_file_path_val, "w") as cqcc_file:
        cqcc_file.write(str(cqcc_fixed_frame_length_val))

    print("Fixed frame length for CQCC (val) saved successfully in Google Drive!")
else:
    print("No valid CQCC data found.")


Total .mat files found: 1505
Processed 1505 files.
Valid CQCC files containing 'cqcc_features' key: 1505

CQCC Data Overview (val):
Total samples: 1505
Min length: 85
Max length: 1239
Mean length: 405.51
Standard Deviation: 170.69
Fixed frame length for CQCC (val) saved successfully in Google Drive!


##test

In [ ]:
import os
import numpy as np
from scipy.io import loadmat

# Example path (update this with your actual path)
cqcc_path = '/content/cqcc_extracted/test'

# Function to load and count CQCC data from .mat files
def load_and_count_cqcc_data(data_path):
    total_files = len([f for f in os.listdir(data_path) if f.endswith('.mat')])
    processed_files = 0
    valid_files = 0
    data_lengths = []

    for file_name in os.listdir(data_path):
        if file_name.endswith('.mat'):  # Check if the file is a .mat file
            processed_files += 1
            try:
                mat_data = loadmat(os.path.join(data_path, file_name))
                if 'cqcc_features' in mat_data:  # Access 'cqcc_features' key
                    valid_files += 1
                    cqcc_features = mat_data['cqcc_features']
                    data_lengths.append(cqcc_features.shape[1])  # Extract the number of time frames
                else:
                    print(f"Warning: 'cqcc_features' key not found in {file_name}")
            except Exception as e:
                print(f"Error processing {file_name}: {e}")

    print(f"Total .mat files found: {total_files}")
    print(f"Processed {processed_files} files.")
    print(f"Valid CQCC files containing 'cqcc_features' key: {valid_files}")
    return data_lengths

# Load and get data lengths for CQCC
cqcc_lengths = load_and_count_cqcc_data(cqcc_path)

# Display statistics for CQCC data
if cqcc_lengths:
    cqcc_mean_length_test = np.mean(cqcc_lengths)
    cqcc_std_dev_length_test = np.std(cqcc_lengths)
    cqcc_fixed_frame_length_test = int(cqcc_mean_length_test + cqcc_std_dev_length_test)
    print("\nCQCC Data Overview (test):")
    print(f"Total samples: {len(cqcc_lengths)}")
    print(f"Min length: {np.min(cqcc_lengths)}")
    print(f"Max length: {np.max(cqcc_lengths)}")
    print(f"Mean length: {np.mean(cqcc_lengths):.2f}")
    print(f"Standard Deviation: {np.std(cqcc_lengths):.2f}")
    # Example path to save in Google Drive
    cqcc_file_path_test = '/content/drive/MyDrive/testing folder for deep learning/cqcc_fixed_frame_length_test.txt'

    # Save the fixed frame length to a text file in Google Drive
    with open(cqcc_file_path_test, "w") as cqcc_file:
        cqcc_file.write(str(cqcc_fixed_frame_length_test))

    print("Fixed frame length for CQCC (test) saved successfully in Google Drive!")
else:
    print("No valid CQCC data found.")


Total .mat files found: 1507
Processed 1507 files.
Valid CQCC files containing 'cqcc_features' key: 1507

CQCC Data Overview (test):
Total samples: 1507
Min length: 68
Max length: 1277
Mean length: 365.54
Standard Deviation: 178.92


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/testing folder for deep learning/cqcc_fixed_frame_length_test.txt'

#Timestamp analysis for pitch and energy

##train

In [ ]:
import pandas as pd
import numpy as np

# Load the CSV file (update 'file_path' with the actual path to your CSV file)
file_path = '/content/drive/MyDrive/EE8223 Group Project/IEMOCAP_Energy_Pitch_Final/train_features_without_nan.csv'
df = pd.read_csv(file_path)

# Assuming your CSV has columns named 'pitch_values' and 'energy_values'
# Extract the data and convert them from string representation of lists to actual lists
df['pitch_values'] = df['pitch_values'].apply(lambda x: [float(i) for i in x.strip('[]').split() if i])
df['energy_values'] = df['energy_values'].apply(lambda x: [float(i) for i in x.strip('[]').split() if i])

# Calculate the number of timeframes for each entry in pitch and energy
df['pitch_timeframes'] = df['pitch_values'].apply(len)
df['energy_timeframes'] = df['energy_values'].apply(len)

# Calculate statistics for pitch timeframes
pitch_mean_length_train = df['pitch_timeframes'].mean()
pitch_std_dev_length_train = df['pitch_timeframes'].std()
pitch_fixed_frame_length_train = int(pitch_mean_length_train + pitch_std_dev_length_train)

# Calculate statistics for energy timeframes
energy_mean_length_train = df['energy_timeframes'].mean()
energy_std_dev_length_train = df['energy_timeframes'].std()
energy_fixed_frame_length_train = int(energy_mean_length_train + energy_std_dev_length_train)

# Print overview statistics for pitch and energy timeframes
print("Pitch Timeframe Overview (train):")
print(f"Total samples: {df['pitch_timeframes'].count()}")
print(f"Min length: {df['pitch_timeframes'].min()}")
print(f"Max length: {df['pitch_timeframes'].max()}")
print(f"Mean length: {df['pitch_timeframes'].mean():.2f}")
print(f"Standard Deviation: {df['pitch_timeframes'].std():.2f}\n")

print("Energy Timeframe Overview (train):")
print(f"Total samples: {df['energy_timeframes'].count()}")
print(f"Min length: {df['energy_timeframes'].min()}")
print(f"Max length: {df['energy_timeframes'].max()}")
print(f"Mean length: {df['energy_timeframes'].mean():.2f}")
print(f"Standard Deviation: {df['energy_timeframes'].std():.2f}")

# Example paths to save in Google Drive
pitch_file_path = '/content/drive/MyDrive/testing folder for deep learning/pitch_fixed_frame_length_train.txt'
energy_file_path = '/content/drive/MyDrive/testing folder for deep learning/energy_fixed_frame_length_train.txt'

# Save the fixed frame lengths to text files in Google Drive
with open(pitch_file_path, "w") as pitch_file:
    pitch_file.write(str(pitch_fixed_frame_length_train))

with open(energy_file_path, "w") as energy_file:
    energy_file.write(str(energy_fixed_frame_length_train))

print("Fixed frame lengths for pitch and energy saved successfully in Google Drive!")


Pitch Timeframe Overview (train):
Total samples: 7027
Min length: 19
Max length: 911
Mean length: 139.31
Standard Deviation: 95.94

Energy Timeframe Overview (train):
Total samples: 7027
Min length: 19
Max length: 911
Mean length: 139.31
Standard Deviation: 95.94
Fixed frame lengths for pitch and energy saved successfully in Google Drive!


##val

In [ ]:
import pandas as pd
import numpy as np

# Load the CSV file (update 'file_path' with the actual path to your CSV file)
file_path = '/content/drive/MyDrive/EE8223 Group Project/IEMOCAP_Energy_Pitch_Final/val_features_without_nan.csv'
df = pd.read_csv(file_path)

# Assuming your CSV has columns named 'pitch_values' and 'energy_values'
# Extract the data and convert them from string representation of lists to actual lists
df['pitch_values'] = df['pitch_values'].apply(lambda x: [float(i) for i in x.strip('[]').split() if i])
df['energy_values'] = df['energy_values'].apply(lambda x: [float(i) for i in x.strip('[]').split() if i])

# Calculate the number of timeframes for each entry in pitch and energy
df['pitch_timeframes'] = df['pitch_values'].apply(len)
df['energy_timeframes'] = df['energy_values'].apply(len)

# Calculate statistics for pitch timeframes (validation)
pitch_mean_length_val = df['pitch_timeframes'].mean()
pitch_std_dev_length_val = df['pitch_timeframes'].std()
pitch_fixed_frame_length_val = int(pitch_mean_length_val + pitch_std_dev_length_val)

# Calculate statistics for energy timeframes (validation)
energy_mean_length_val = df['energy_timeframes'].mean()
energy_std_dev_length_val = df['energy_timeframes'].std()
energy_fixed_frame_length_val = int(energy_mean_length_val + energy_std_dev_length_val)

# Example paths to save in Google Drive
pitch_file_path_val = '/content/drive/MyDrive/testing folder for deep learning/pitch_fixed_frame_length_val.txt'
energy_file_path_val = '/content/drive/MyDrive/testing folder for deep learning/energy_fixed_frame_length_val.txt'


# Print overview statistics for pitch and energy timeframes
print("Pitch Timeframe Overview (val):")
print(f"Total samples: {df['pitch_timeframes'].count()}")
print(f"Min length: {df['pitch_timeframes'].min()}")
print(f"Max length: {df['pitch_timeframes'].max()}")
print(f"Mean length: {df['pitch_timeframes'].mean():.2f}")
print(f"Standard Deviation: {df['pitch_timeframes'].std():.2f}\n")

print("Energy Timeframe Overview (val):")
print(f"Total samples: {df['energy_timeframes'].count()}")
print(f"Min length: {df['energy_timeframes'].min()}")
print(f"Max length: {df['energy_timeframes'].max()}")
print(f"Mean length: {df['energy_timeframes'].mean():.2f}")
print(f"Standard Deviation: {df['energy_timeframes'].std():.2f}")

# Save the fixed frame lengths to text files in Google Drive
with open(pitch_file_path_val, "w") as pitch_file:
    pitch_file.write(str(pitch_fixed_frame_length_val))

with open(energy_file_path_val, "w") as energy_file:
    energy_file.write(str(energy_fixed_frame_length_val))

print("Fixed frame lengths for pitch and energy (validation) saved successfully in Google Drive!")

Pitch Timeframe Overview (val):
Total samples: 1505
Min length: 21
Max length: 998
Mean length: 140.71
Standard Deviation: 97.75

Energy Timeframe Overview (val):
Total samples: 1505
Min length: 21
Max length: 998
Mean length: 140.71
Standard Deviation: 97.75
Fixed frame lengths for pitch and energy (validation) saved successfully in Google Drive!


##test

In [ ]:
import pandas as pd
import numpy as np

# Load the CSV file (update 'file_path' with the actual path to your CSV file)
file_path = '/content/drive/MyDrive/EE8223 Group Project/IEMOCAP_Energy_Pitch_Final/test_features_without_nan.csv'
df = pd.read_csv(file_path)

# Assuming your CSV has columns named 'pitch_values' and 'energy_values'
# Extract the data and convert them from string representation of lists to actual lists
# Added error handling to replace '...' with 0.001
df['pitch_values'] = df['pitch_values'].apply(lambda x: [float(i) if i != '...' else 0.001 for i in x.strip('[]').split() if i])
df['energy_values'] = df['energy_values'].apply(lambda x: [float(i) if i != '...' else 0.001 for i in x.strip('[]').split() if i])

# Calculate the number of timeframes for each entry in pitch and energy
df['pitch_timeframes'] = df['pitch_values'].apply(len)
df['energy_timeframes'] = df['energy_values'].apply(len)

# Calculate statistics for pitch timeframes (test)
pitch_mean_length_test = df['pitch_timeframes'].mean()
pitch_std_dev_length_test = df['pitch_timeframes'].std()
pitch_fixed_frame_length_test = int(pitch_mean_length_test + pitch_std_dev_length_test)

# Calculate statistics for energy timeframes (test)
energy_mean_length_test = df['energy_timeframes'].mean()
energy_std_dev_length_test = df['energy_timeframes'].std()
energy_fixed_frame_length_test = int(energy_mean_length_test + energy_std_dev_length_test)

# Example paths to save in Google Drive
pitch_file_path_test = '/content/drive/MyDrive/testing folder for deep learning/pitch_fixed_frame_length_test.txt'
energy_file_path_test = '/content/drive/MyDrive/testing folder for deep learning/energy_fixed_frame_length_test.txt'

# Print overview statistics for pitch and energy timeframes
print("Pitch Timeframe Overview:")
print(f"Total samples: {df['pitch_timeframes'].count()}")
print(f"Min length: {df['pitch_timeframes'].min()}")
print(f"Max length: {df['pitch_timeframes'].max()}")
print(f"Mean length: {df['pitch_timeframes'].mean():.2f}")
print(f"Standard Deviation: {df['pitch_timeframes'].std():.2f}\n")

print("Energy Timeframe Overview (test):")
print(f"Total samples: {df['energy_timeframes'].count()}")
print(f"Min length: {df['energy_timeframes'].min()}")
print(f"Max length: {df['energy_timeframes'].max()}")
print(f"Mean length: {df['energy_timeframes'].mean():.2f}")
print(f"Standard Deviation: {df['energy_timeframes'].std():.2f}")

# Save the fixed frame lengths to text files in Google Drive
with open(pitch_file_path_test, "w") as pitch_file:
    pitch_file.write(str(pitch_fixed_frame_length_test))

with open(energy_file_path_test, "w") as energy_file:
    energy_file.write(str(energy_fixed_frame_length_test))

print("Fixed frame lengths for pitch and energy (test) saved successfully in Google Drive!")


Pitch Timeframe Overview:
Total samples: 1507
Min length: 7
Max length: 746
Mean length: 140.99
Standard Deviation: 89.97

Energy Timeframe Overview (test):
Total samples: 1507
Min length: 7
Max length: 746
Mean length: 140.99
Standard Deviation: 89.97
Fixed frame lengths for pitch and energy (test) saved successfully in Google Drive!


In [ ]:
# Fixed frame length for embeddings
embedding_fixed_frame_length = 1024

# Example path to save in Google Drive
embedding_file_path = '/content/drive/MyDrive/testing folder for deep learning/embedding_fixed_frame_length.txt'

# Save the fixed frame length to a text file in Google Drive
with open(embedding_file_path, "w") as embedding_file:
    embedding_file.write(str(embedding_fixed_frame_length))

print("Fixed frame length for embeddings saved successfully in Google Drive!")


Fixed frame length for embeddings saved successfully in Google Drive!


# Padding and Truncating

# MFCC Train

In [ ]:
import os
import numpy as np

# Function to pad or truncate data to the fixed length
def pad_or_truncate(data, fixed_length):
    if data.shape[1] > fixed_length:
        # Truncate if the sequence is longer than the fixed length
        return data[:, :fixed_length]
    elif data.shape[1] < fixed_length:
        # Pad with zeros if the sequence is shorter than the fixed length
        padding = np.zeros((data.shape[0], fixed_length - data.shape[1]))
        return np.concatenate((data, padding), axis=1)
    else:
        # Return the data unchanged if it already matches the fixed length
        return data

# Example path to your MFCC train data and the fixed frame length
mfcc_path_train = '/content/mfcc_extracted/train'

# Set the fixed frame length (as you've calculated)
mfcc_fixed_frame_length_train = int((1/3)*(mfcc_mean_length_train + mfcc_mean_length_val + mfcc_mean_length_test) + max(mfcc_std_dev_length_test, mfcc_std_dev_length_train, mfcc_std_dev_length_val))

# Function to load, pad/truncate, and store MFCC train data with file names
def process_and_save_mfcc_train_data_with_names(data_path, fixed_length, save_path):
    data_list = []
    name_list = []

    # Load and process each .npy file
    for file_name in os.listdir(data_path):
        if file_name.endswith('.npy'):
            # Load the MFCC data
            data = np.load(os.path.join(data_path, file_name))

            # Pad or truncate the data
            data = pad_or_truncate(data, fixed_length)

            # Append to the lists
            data_list.append(data)
            name_list.append(file_name)

    # Save the processed data and file names as a .npz file
    np.savez(save_path, data=np.array(data_list), file_names=np.array(name_list))
    print("Processed MFCC train data and file names saved successfully!")

# Path to save the processed MFCC train data and file names
processed_mfcc_train_path = '/content/drive/MyDrive/testing folder for deep learning/fixed_length_data/processed_mfcc_train_with_names.npz'

# Process and save the MFCC train data with file names
process_and_save_mfcc_train_data_with_names(mfcc_path_train, mfcc_fixed_frame_length_train, processed_mfcc_train_path)


Processed MFCC train data and file names saved successfully!


# MFCC Val

In [ ]:
import os
import numpy as np

# Function to pad or truncate data to the fixed length
def pad_or_truncate(data, fixed_length):
    if data.shape[1] > fixed_length:
        # Truncate if the sequence is longer than the fixed length
        return data[:, :fixed_length]
    elif data.shape[1] < fixed_length:
        # Pad with zeros if the sequence is shorter than the fixed length
        padding = np.zeros((data.shape[0], fixed_length - data.shape[1]))
        return np.concatenate((data, padding), axis=1)
    else:
        # Return the data unchanged if it already matches the fixed length
        return data

# Example path to your MFCC validation data and the fixed frame length
mfcc_path_val = '/content/mfcc_extracted/val'

# Set the fixed frame length (as you've calculated)
mfcc_fixed_frame_length_val = int((1/3)*(mfcc_mean_length_train + mfcc_mean_length_val + mfcc_mean_length_test) + max(mfcc_std_dev_length_test, mfcc_std_dev_length_train, mfcc_std_dev_length_val))

# Function to load, pad/truncate, and store MFCC validation data with file names
def process_and_save_mfcc_val_data_with_names(data_path, fixed_length, save_path):
    data_list = []
    name_list = []

    # Load and process each .npy file
    for file_name in os.listdir(data_path):
        if file_name.endswith('.npy'):
            # Load the MFCC data
            data = np.load(os.path.join(data_path, file_name))

            # Pad or truncate the data
            data = pad_or_truncate(data, fixed_length)

            # Append to the lists
            data_list.append(data)
            name_list.append(file_name)

    # Save the processed data and file names as a .npz file
    np.savez(save_path, data=np.array(data_list), file_names=np.array(name_list))
    print("Processed MFCC validation data and file names saved successfully!")

# Path to save the processed MFCC validation data and file names
processed_mfcc_val_path = '/content/drive/MyDrive/testing folder for deep learning/fixed_length_data/processed_mfcc_val_with_names.npz'

# Process and save the MFCC validation data with file names
process_and_save_mfcc_val_data_with_names(mfcc_path_val, mfcc_fixed_frame_length_val, processed_mfcc_val_path)


Processed MFCC validation data and file names saved successfully!


# MFCC Test

In [ ]:
import os
import numpy as np

# Function to pad or truncate data to the fixed length
def pad_or_truncate(data, fixed_length):
    if data.shape[1] > fixed_length:
        # Truncate if the sequence is longer than the fixed length
        return data[:, :fixed_length]
    elif data.shape[1] < fixed_length:
        # Pad with zeros if the sequence is shorter than the fixed length
        padding = np.zeros((data.shape[0], fixed_length - data.shape[1]))
        return np.concatenate((data, padding), axis=1)
    else:
        # Return the data unchanged if it already matches the fixed length
        return data

# Example path to your MFCC test data and the fixed frame length
mfcc_path_test = '/content/mfcc_extracted/test'

# Set the fixed frame length (as you've calculated)
mfcc_fixed_frame_length_test = int((1/3)*(mfcc_mean_length_train + mfcc_mean_length_val + mfcc_mean_length_test) + max(mfcc_std_dev_length_test, mfcc_std_dev_length_train, mfcc_std_dev_length_val))

# Function to load, pad/truncate, and store MFCC test data with file names
def process_and_save_mfcc_test_data_with_names(data_path, fixed_length, save_path):
    data_list = []
    name_list = []

    # Load and process each .npy file
    for file_name in os.listdir(data_path):
        if file_name.endswith('.npy'):
            # Load the MFCC data
            data = np.load(os.path.join(data_path, file_name))

            # Pad or truncate the data
            data = pad_or_truncate(data, fixed_length)

            # Append to the lists
            data_list.append(data)
            name_list.append(file_name)

    # Save the processed data and file names as a .npz file
    np.savez(save_path, data=np.array(data_list), file_names=np.array(name_list))
    print("Processed MFCC test data and file names saved successfully!")

# Path to save the processed MFCC test data and file names
processed_mfcc_test_path = '/content/drive/MyDrive/testing folder for deep learning/fixed_length_data/processed_mfcc_test_with_names.npz'

# Process and save the MFCC test data with file names
process_and_save_mfcc_test_data_with_names(mfcc_path_test, mfcc_fixed_frame_length_test, processed_mfcc_test_path)


Processed MFCC test data and file names saved successfully!


# CQCC Train

In [ ]:
import os
import numpy as np

# Function to pad or truncate data to the fixed length
def pad_or_truncate(data, fixed_length):
    if data.shape[1] > fixed_length:
        # Truncate if the sequence is longer than the fixed length
        return data[:, :fixed_length]
    elif data.shape[1] < fixed_length:
        # Pad with zeros if the sequence is shorter than the fixed length
        padding = np.zeros((data.shape[0], fixed_length - data.shape[1]))
        return np.concatenate((data, padding), axis=1)
    else:
        # Return the data unchanged if it already matches the fixed length
        return data

# Example path to your CQCC train data and the fixed frame length
cqcc_path_train = '/content/cqcc_extracted/train_wav'

# Set the fixed frame length (as you've calculated)
cqcc_fixed_frame_length_train = int((1/3)*(cqcc_mean_length_train + cqcc_mean_length_val + cqcc_mean_length_test) + max(cqcc_std_dev_length_test, cqcc_std_dev_length_train, cqcc_std_dev_length_val))

# Function to load, pad/truncate, and store CQCC train data with file names
def process_and_save_cqcc_train_data_with_names(data_path, fixed_length, save_path):
    data_list = []
    name_list = []

    # Load and process each .npy file
    for file_name in os.listdir(data_path):
        if file_name.endswith('.npy'):
            # Load the CQCC data
            data = np.load(os.path.join(data_path, file_name))

            # Pad or truncate the data
            data = pad_or_truncate(data, fixed_length)

            # Append to the lists
            data_list.append(data)
            name_list.append(file_name)

    # Save the processed data and file names as a .npz file
    np.savez(save_path, data=np.array(data_list), file_names=np.array(name_list))
    print("Processed CQCC train data and file names saved successfully!")

# Path to save the processed CQCC train data and file names
processed_cqcc_train_path = '/content/drive/MyDrive/testing folder for deep learning/fixed_length_data/processed_cqcc_train_with_names.npz'

# Process and save the CQCC train data with file names
process_and_save_cqcc_train_data_with_names(cqcc_path_train, cqcc_fixed_frame_length_train, processed_cqcc_train_path)


Processed CQCC train data and file names saved successfully!


# CQCC VAL

In [ ]:
import os
import numpy as np

# Function to pad or truncate data to the fixed length
def pad_or_truncate(data, fixed_length):
    if data.shape[1] > fixed_length:
        # Truncate if the sequence is longer than the fixed length
        return data[:, :fixed_length]
    elif data.shape[1] < fixed_length:
        # Pad with zeros if the sequence is shorter than the fixed length
        padding = np.zeros((data.shape[0], fixed_length - data.shape[1]))
        return np.concatenate((data, padding), axis=1)
    else:
        # Return the data unchanged if it already matches the fixed length
        return data

# Example path to your CQCC validation data and the fixed frame length
cqcc_path_val = '/content/cqcc_extracted/val_wav'

# Set the fixed frame length (as you've calculated)
cqcc_fixed_frame_length_val = int((1/3)*(cqcc_mean_length_train + cqcc_mean_length_val + cqcc_mean_length_test) + max(cqcc_std_dev_length_test, cqcc_std_dev_length_train, cqcc_std_dev_length_val))

# Function to load, pad/truncate, and store CQCC validation data with file names
def process_and_save_cqcc_val_data_with_names(data_path, fixed_length, save_path):
    data_list = []
    name_list = []

    # Load and process each .npy file
    for file_name in os.listdir(data_path):
        if file_name.endswith('.npy'):
            # Load the CQCC data
            data = np.load(os.path.join(data_path, file_name))

            # Pad or truncate the data
            data = pad_or_truncate(data, fixed_length)

            # Append to the lists
            data_list.append(data)
            name_list.append(file_name)

    # Save the processed data and file names as a .npz file
    np.savez(save_path, data=np.array(data_list), file_names=np.array(name_list))
    print("Processed CQCC validation data and file names saved successfully!")

# Path to save the processed CQCC validation data and file names
processed_cqcc_val_path = '/content/drive/MyDrive/testing folder for deep learning/fixed_length_data/processed_cqcc_val_with_names.npz'

# Process and save the CQCC validation data with file names
process_and_save_cqcc_val_data_with_names(cqcc_path_val, cqcc_fixed_frame_length_val, processed_cqcc_val_path)


Processed CQCC validation data and file names saved successfully!


# CQCC Test

In [ ]:
import os
import numpy as np

# Function to pad or truncate data to the fixed length
def pad_or_truncate(data, fixed_length):
    if data.shape[1] > fixed_length:
        # Truncate if the sequence is longer than the fixed length
        return data[:, :fixed_length]
    elif data.shape[1] < fixed_length:
        # Pad with zeros if the sequence is shorter than the fixed length
        padding = np.zeros((data.shape[0], fixed_length - data.shape[1]))
        return np.concatenate((data, padding), axis=1)
    else:
        # Return the data unchanged if it already matches the fixed length
        return data

# Example path to your CQCC test data and the fixed frame length
cqcc_path_test = '/content/cqcc_extracted/test'

# Set the fixed frame length (as you've calculated)
cqcc_fixed_frame_length_test = int((1/3)*(cqcc_mean_length_train + cqcc_mean_length_val + cqcc_mean_length_test) + max(cqcc_std_dev_length_test, cqcc_std_dev_length_train, cqcc_std_dev_length_val))

# Function to load, pad/truncate, and store CQCC test data with file names
def process_and_save_cqcc_test_data_with_names(data_path, fixed_length, save_path):
    data_list = []
    name_list = []

    # Load and process each .npy file
    for file_name in os.listdir(data_path):
        if file_name.endswith('.npy'):
            # Load the CQCC data
            data = np.load(os.path.join(data_path, file_name))

            # Pad or truncate the data
            data = pad_or_truncate(data, fixed_length)

            # Append to the lists
            data_list.append(data)
            name_list.append(file_name)

    # Save the processed data and file names as a .npz file
    np.savez(save_path, data=np.array(data_list), file_names=np.array(name_list))
    print("Processed CQCC test data and file names saved successfully!")

# Path to save the processed CQCC test data and file names
processed_cqcc_test_path = '/content/drive/MyDrive/testing folder for deep learning/fixed_length_data/processed_cqcc_test_with_names.npz'

# Process and save the CQCC test data with file names
process_and_save_cqcc_test_data_with_names(cqcc_path_test, cqcc_fixed_frame_length_test, processed_cqcc_test_path)


Processed CQCC test data and file names saved successfully!


# Pitch Train

In [ ]:
import pandas as pd
import numpy as np

# Function to read the fixed frame length from a text file
def read_fixed_frame_length(file_path):
    with open(file_path, "r") as file:
        return int(file.read().strip())

# Function to pad or truncate data to the fixed length
def pad_or_truncate(data, fixed_length):
    if len(data) > fixed_length:
        # Truncate if the sequence is longer than the fixed length
        return data[:fixed_length]
    elif len(data) < fixed_length:
        # Pad with zeros if the sequence is shorter than the fixed length
        return np.pad(data, (0, fixed_length - len(data)), 'constant')
    else:
        # Return the data unchanged if it already matches the fixed length
        return data

# Example path to your Pitch train data and the fixed frame length file
pitch_train_csv_path = '/content/drive/MyDrive/EE8223 Group Project/IEMOCAP_Energy_Pitch_Final/train_features_without_nan.csv'
pitch_fixed_frame_length_file_path_train = '/content/drive/MyDrive/testing folder for deep learning/pitch_fixed_frame_length_train.txt'

# Read the fixed frame length from the text file
pitch_fixed_frame_length_train = int((1/3)*(pitch_mean_length_train + pitch_mean_length_val +pitch_mean_length_test) + max(pitch_std_dev_length_test,pitch_std_dev_length_train,pitch_std_dev_length_val))


# Load the Pitch train data from the CSV file
df_train = pd.read_csv(pitch_train_csv_path)

# Convert the 'pitch_values' from string representation to a list of floats
df_train['pitch_values'] = df_train['pitch_values'].apply(lambda x: [float(i) for i in x.strip('[]').split() if i])

# Pad or truncate each sequence in the 'pitch_values' column
processed_pitch_train = np.array([pad_or_truncate(np.array(values), pitch_fixed_frame_length_train) for values in df_train['pitch_values']])

# Path to save the processed Pitch train data
processed_pitch_train_path = '/content/drive/MyDrive/testing folder for deep learning/fixed_length_data/Pitch_Fixed/processed_pitch_train.npy'

# Save the processed Pitch train data
np.save(processed_pitch_train_path, processed_pitch_train)
print("Processed Pitch train data saved successfully in Google Drive!")


Processed Pitch train data saved successfully in Google Drive!


# Pitch Val

In [ ]:
import pandas as pd
import numpy as np

# Function to read the fixed frame length from a text file
def read_fixed_frame_length(file_path):
    with open(file_path, "r") as file:
        return int(file.read().strip())

# Function to pad or truncate data to the fixed length
def pad_or_truncate(data, fixed_length):
    if len(data) > fixed_length:
        # Truncate if the sequence is longer than the fixed length
        return data[:fixed_length]
    elif len(data) < fixed_length:
        # Pad with zeros if the sequence is shorter than the fixed length
        return np.pad(data, (0, fixed_length - len(data)), 'constant')
    else:
        # Return the data unchanged if it already matches the fixed length
        return data

# Example path to your Pitch validation data and the fixed frame length file
pitch_val_csv_path = '/content/drive/MyDrive/EE8223 Group Project/IEMOCAP_Energy_Pitch_Final/val_features_without_nan.csv'
pitch_fixed_frame_length_file_path_val = '/content/drive/MyDrive/testing folder for deep learning/pitch_fixed_frame_length_val.txt'

# Read the fixed frame length from the text file
pitch_fixed_frame_length_val =int((1/3)*(pitch_mean_length_train + pitch_mean_length_val +pitch_mean_length_test) + max(pitch_std_dev_length_test,pitch_std_dev_length_train,pitch_std_dev_length_val))

# Load the Pitch validation data from the CSV file
df_val = pd.read_csv(pitch_val_csv_path)

# Convert the 'pitch_values' from string representation to a list of floats
df_val['pitch_values'] = df_val['pitch_values'].apply(lambda x: [float(i) for i in x.strip('[]').split() if i])

# Pad or truncate each sequence in the 'pitch_values' column
processed_pitch_val = np.array([pad_or_truncate(np.array(values), pitch_fixed_frame_length_val) for values in df_val['pitch_values']])

# Path to save the processed Pitch validation data
processed_pitch_val_path = '/content/drive/MyDrive/testing folder for deep learning/fixed_length_data/Pitch_Fixed/processed_pitch_val.npy'

# Save the processed Pitch validation data
np.save(processed_pitch_val_path, processed_pitch_val)
print("Processed Pitch validation data saved successfully in Google Drive!")


Processed Pitch validation data saved successfully in Google Drive!


# Pitch Test

In [ ]:
import pandas as pd
import numpy as np

# Function to read the fixed frame length from a text file
def read_fixed_frame_length(file_path):
    with open(file_path, "r") as file:
        return int(file.read().strip())

# Function to pad or truncate data to the fixed length
def pad_or_truncate(data, fixed_length):
    if len(data) > fixed_length:
        # Truncate if the sequence is longer than the fixed length
        return data[:fixed_length]
    elif len(data) < fixed_length:
        # Pad with zeros if the sequence is shorter than the fixed length
        return np.pad(data, (0, fixed_length - len(data)), 'constant')
    else:
        # Return the data unchanged if it already matches the fixed length
        return data

# Example path to your Pitch test data and the fixed frame length file
pitch_test_csv_path = '/content/drive/MyDrive/EE8223 Group Project/IEMOCAP_Energy_Pitch_Final/test_features_without_nan.csv'
pitch_fixed_frame_length_file_path_test = '/content/drive/MyDrive/testing folder for deep learning/pitch_fixed_frame_length_test.txt'

# Read the fixed frame length from the text file
pitch_fixed_frame_length_test = int((1/3)*(pitch_mean_length_train + pitch_mean_length_val +pitch_mean_length_test) + max(pitch_std_dev_length_test,pitch_std_dev_length_train,pitch_std_dev_length_val))
# Load the Pitch test data from the CSV file
df_test = pd.read_csv(pitch_test_csv_path)

# Convert the 'pitch_values' from string representation to a list of floats
# Replace '...' with 0.001 and handle NaN values
df_test['pitch_values'] = df_test['pitch_values'].apply(
    lambda x: [float(i) if i != '...' else 0.001 for i in x.strip('[]').split() if i]
)

# Handle potential NaN values
df_test['pitch_values'] = df_test['pitch_values'].apply(
    lambda x: [0.001 if pd.isna(i) else i for i in x]
)

# Pad or truncate each sequence in the 'pitch_values' column
processed_pitch_test = np.array([pad_or_truncate(np.array(values), pitch_fixed_frame_length_test) for values in df_test['pitch_values']])

# Path to save the processed Pitch test data
processed_pitch_test_path = '/content/drive/MyDrive/testing folder for deep learning/fixed_length_data/Pitch_Fixed/processed_pitch_test.npy'

# Save the processed Pitch test data
np.save(processed_pitch_test_path, processed_pitch_test)
print("Processed Pitch test data saved successfully in Google Drive!")



Processed Pitch test data saved successfully in Google Drive!


# Energy Train

In [ ]:
import pandas as pd
import numpy as np

# Function to read the fixed frame length from a text file
def read_fixed_frame_length(file_path):
    with open(file_path, "r") as file:
        return int(file.read().strip())

# Function to pad or truncate data to the fixed length
def pad_or_truncate(data, fixed_length):
    if len(data) > fixed_length:
        # Truncate if the sequence is longer than the fixed length
        return data[:fixed_length]
    elif len(data) < fixed_length:
        # Pad with zeros if the sequence is shorter than the fixed length
        return np.pad(data, (0, fixed_length - len(data)), 'constant')
    else:
        # Return the data unchanged if it already matches the fixed length
        return data

# Example path to your Energy train data and the fixed frame length file
energy_train_csv_path = '/content/drive/MyDrive/EE8223 Group Project/IEMOCAP_Energy_Pitch_Final/train_features.csv'
energy_fixed_frame_length_file_path_train = '/content/drive/MyDrive/testing folder for deep learning/energy_fixed_frame_length_train.txt'

# Read the fixed frame length from the text file
energy_fixed_frame_length_train = int((1/3)*(energy_mean_length_train + energy_mean_length_val +energy_mean_length_test) + max(energy_std_dev_length_test,energy_std_dev_length_train,energy_std_dev_length_val))

# Load the Energy train data from the CSV file
df_train = pd.read_csv(energy_train_csv_path)

# Convert the 'energy_values' from string representation to a list of floats
df_train['energy_values'] = df_train['energy_values'].apply(
    lambda x: [float(i) for i in x.strip('[]').split() if i]
)

# Pad or truncate each sequence in the 'energy_values' column
processed_energy_train = np.array([pad_or_truncate(np.array(values), energy_fixed_frame_length_train) for values in df_train['energy_values']])

# Path to save the processed Energy train data
processed_energy_train_path = '/content/drive/MyDrive/testing folder for deep learning/fixed_length_data/processed_energy_train.npy'

# Save the processed Energy train data
np.save(processed_energy_train_path, processed_energy_train)
print("Processed Energy train data saved successfully in Google Drive!")


Processed Energy train data saved successfully in Google Drive!


# Energy Val

In [ ]:
import pandas as pd
import numpy as np

# Function to read the fixed frame length from a text file
def read_fixed_frame_length(file_path):
    with open(file_path, "r") as file:
        return int(file.read().strip())

# Function to pad or truncate data to the fixed length
def pad_or_truncate(data, fixed_length):
    if len(data) > fixed_length:
        # Truncate if the sequence is longer than the fixed length
        return data[:fixed_length]
    elif len(data) < fixed_length:
        # Pad with zeros if the sequence is shorter than the fixed length
        return np.pad(data, (0, fixed_length - len(data)), 'constant')
    else:
        # Return the data unchanged if it already matches the fixed length
        return data

# Example path to your Energy validation data and the fixed frame length file
energy_val_csv_path = '/content/drive/MyDrive/EE8223 Group Project/IEMOCAP_Energy_Pitch_Final/val_features.csv'
energy_fixed_frame_length_file_path_val = '/content/drive/MyDrive/testing folder for deep learning/energy_fixed_frame_length_val.txt'

# Read the fixed frame length from the text file
energy_fixed_frame_length_val = int((1/3)*(energy_mean_length_train + energy_mean_length_val +energy_mean_length_test) + max(energy_std_dev_length_test,energy_std_dev_length_train,energy_std_dev_length_val))

# Load the Energy validation data from the CSV file
df_val = pd.read_csv(energy_val_csv_path)

# Convert the 'energy_values' from string representation to a list of floats
df_val['energy_values'] = df_val['energy_values'].apply(
    lambda x: [float(i) for i in x.strip('[]').split() if i]
)

# Pad or truncate each sequence in the 'energy_values' column
processed_energy_val = np.array([pad_or_truncate(np.array(values), energy_fixed_frame_length_val) for values in df_val['energy_values']])

# Path to save the processed Energy validation data
processed_energy_val_path = '/content/drive/MyDrive/testing folder for deep learning/fixed_length_data/processed_energy_val.npy'

# Save the processed Energy validation data
np.save(processed_energy_val_path, processed_energy_val)
print("Processed Energy validation data saved successfully in Google Drive!")


Processed Energy validation data saved successfully in Google Drive!


# Energy Test

In [ ]:
import pandas as pd
import numpy as np

# Function to read the fixed frame length from a text file
def read_fixed_frame_length(file_path):
    with open(file_path, "r") as file:
        return int(file.read().strip())

# Function to pad or truncate data to the fixed length
def pad_or_truncate(data, fixed_length):
    if len(data) > fixed_length:
        # Truncate if the sequence is longer than the fixed length
        return data[:fixed_length]
    elif len(data) < fixed_length:
        # Pad with zeros if the sequence is shorter than the fixed length
        return np.pad(data, (0, fixed_length - len(data)), 'constant')
    else:
        # Return the data unchanged if it already matches the fixed length
        return data

# Example path to your Energy test data and the fixed frame length file
energy_test_csv_path = '/content/drive/MyDrive/EE8223 Group Project/IEMOCAP_Energy_Pitch_Final/test_features_new.csv'
energy_fixed_frame_length_file_path_test = '/content/drive/MyDrive/testing folder for deep learning/energy_fixed_frame_length_test.txt'

# Read the fixed frame length from the text file
energy_fixed_frame_length_test = int((1/3)*(energy_mean_length_train + energy_mean_length_val +energy_mean_length_test) + max(energy_std_dev_length_test,energy_std_dev_length_train,energy_std_dev_length_val))
# Load the Energy test data from the CSV file
df_test = pd.read_csv(energy_test_csv_path)

# Convert the 'energy_values' from string representation to a list of floats
# Replace '...' with 0.001 and handle NaN values
df_test['energy_values'] = df_test['energy_values'].apply(
    lambda x: [float(i) if i != '...' else 0.001 for i in x.strip('[]').split() if i]
)

# Handle potential NaN values
df_test['energy_values'] = df_test['energy_values'].apply(
    lambda x: [0.001 if pd.isna(i) else i for i in x]
)

# Pad or truncate each sequence in the 'energy_values' column
processed_energy_test = np.array([pad_or_truncate(np.array(values), energy_fixed_frame_length_test) for values in df_test['energy_values']])

# Path to save the processed Energy test data
processed_energy_test_path = '/content/drive/MyDrive/testing folder for deep learning/fixed_length_data/processed_energy_test.npy'

# Save the processed Energy test data
np.save(processed_energy_test_path, processed_energy_test)
print("Processed Energy test data saved successfully in Google Drive!")



Processed Energy test data saved successfully in Google Drive!


# Embedding Train, Val, and Test

In [ ]:
import pandas as pd
import numpy as np

# Function to clean and convert the 'embedding' column from string representation to a list of floats
def process_embedding_column(df, column_name='embedding'):
    df[column_name] = df[column_name].apply(
        lambda x: [float(i.strip()) for i in x.strip('[]').split(',') if i.strip()]
    )
    return np.array(df[column_name].tolist())

# Paths to your Embedding data CSV files
embedding_train_csv_path = '/content/drive/MyDrive/IEMOCAP_embeddings/iemocap_embeddings_train.csv'
embedding_val_csv_path = '/content/drive/MyDrive/IEMOCAP_embeddings/iemocap_embeddings_validate.csv'
embedding_test_csv_path = '/content/drive/MyDrive/IEMOCAP_embeddings/iemocap_embeddings_test.csv'

# Paths to save the processed Embedding data
processed_train_embedding_path = '/content/drive/MyDrive/testing folder for deep learning/processed_embedding_train.npy'
processed_val_embedding_path = '/content/drive/MyDrive/testing folder for deep learning/processed_embedding_val.npy'
processed_test_embedding_path = '/content/drive/MyDrive/testing folder for deep learning/processed_embedding_test.npy'

# Process and save Embedding Train Data
df_train_embedding = pd.read_csv(embedding_train_csv_path)
processed_train_embedding = process_embedding_column(df_train_embedding)
np.save(processed_train_embedding_path, processed_train_embedding)
print("Processed Embedding train data saved successfully in Google Drive!")

# Process and save Embedding Validation Data
df_val_embedding = pd.read_csv(embedding_val_csv_path)
processed_val_embedding = process_embedding_column(df_val_embedding)
np.save(processed_val_embedding_path, processed_val_embedding)
print("Processed Embedding validation data saved successfully in Google Drive!")

# Process and save Embedding Test Data
df_test_embedding = pd.read_csv(embedding_test_csv_path)
processed_test_embedding = process_embedding_column(df_test_embedding)
np.save(processed_test_embedding_path, processed_test_embedding)
print("Processed Embedding test data saved successfully in Google Drive!")



Processed Embedding train data saved successfully in Google Drive!
Processed Embedding validation data saved successfully in Google Drive!
Processed Embedding test data saved successfully in Google Drive!
